<a href="https://colab.research.google.com/github/k3ntut/SeaOfFtars/blob/main/Identifying_Generated_Artworks_with_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, Callback
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
import os
import matplotlib.pyplot as plt

# Define image size
image_size = 224
imageFolder = 'D:/EfficientNetV2/dataset50/train/'
CLASSES = os.listdir(imageFolder)
num_classes = len(CLASSES)
print(CLASSES)
print(num_classes)

# Define data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_folder = 'D:/EfficientNetV2/dataset50/train/'
test_folder = 'D:/EfficientNetV2/dataset50/test/'

train_generator = train_datagen.flow_from_directory(
    train_folder,
    target_size=(image_size, image_size),
    batch_size=64,
    class_mode='categorical',
    color_mode='rgb',
    shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    test_folder,
    target_size=(image_size, image_size),
    batch_size=64,
    class_mode='categorical',
    color_mode='rgb'
)

# Define and compile the model
from tensorflow.keras.applications import VGG16

base_model = VGG16(weights='imagenet', input_shape=(image_size, image_size, 3), include_top=False)
base_model.trainable = True

model = tf.keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

adam_opt = tf.keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=adam_opt, loss='categorical_crossentropy', metrics=['accuracy'])

# Custom callback to compute F1 score at the end of each epoch
class F1ScoreCallback(Callback):
    def __init__(self, validation_data):
        super(F1ScoreCallback, self).__init__()
        self.validation_data = validation_data
        self.f1_scores = []

    def on_epoch_end(self, epoch, logs=None):
        val_gen = self.validation_data
        y_pred = self.model.predict(val_gen)
        y_pred_labels = np.argmax(y_pred, axis=1)
        y_true = val_gen.classes
        f1 = f1_score(y_true, y_pred_labels, average='macro')
        self.f1_scores.append(f1)
        print(f' — val_f1: {f1:.4f}')
        logs['val_f1'] = f1  # Adding F1 score to logs

# Define callbacks
epochs = 50
best_model_file = 'D:/EfficientNetV2/VGG16-21Categories.keras'

callbacks = [
    ModelCheckpoint(best_model_file, verbose=1, save_best_only=True, monitor='val_accuracy'),
    ReduceLROnPlateau(monitor='val_accuracy', patience=5, factor=0.1, min_lr=1e-6, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=3, verbose=1),
    F1ScoreCallback(validation_data=test_generator)  # Add custom F1 score callback
]

# Train the model
try:
    result = model.fit(
        train_generator, epochs=epochs, validation_data=test_generator, callbacks=callbacks
    )

    # Display results
    best_val_acc_epoch = np.argmax(result.history['val_accuracy'])
    best_val_acc = result.history['val_accuracy'][best_val_acc_epoch]
    print("Best validation accuracy: " + str(best_val_acc))

    plt.plot(result.history['accuracy'], label='Train Accuracy')
    plt.plot(result.history['val_accuracy'], label='Validation Accuracy')
    plt.legend()
    plt.show()

    plt.plot(result.history['loss'], label='Train Loss')
    plt.plot(result.history['val_loss'], label='Validation Loss')
    plt.legend()
    plt.show()

    # Plot F1 score history
    plt.plot(callbacks[-1].f1_scores, label='Validation F1 Score')
    plt.legend()
    plt.show()

except Exception as e:
    print(f"Error during training: {e}")

# Predict the labels for the test set
y_pred = model.predict(test_generator)
y_pred_labels = np.argmax(y_pred, axis=1)  # Convert probabilities to class labels

# Get the true labels from the test generator
y_true = test_generator.classes

# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred_labels)
print(f'Accuracy on Test Set: {accuracy:.4f}')

# Calculate F1 score
f1score = f1_score(y_true, y_pred_labels, average='macro')
print(f'F1 Score on Test Set: {f1score:.4f}')

In [ ]:
# Train the model
try:
    result = model.fit(
        train_generator, epochs=epochs, validation_data=test_generator, callbacks=callbacks
    )

    # Display results
    best_val_acc_epoch = np.argmax(result.history['val_accuracy'])
    best_val_acc = result.history['val_accuracy'][best_val_acc_epoch]
    print("Best validation accuracy: " + str(best_val_acc))

    plt.plot(result.history['accuracy'], label='Train Accuracy')
    plt.plot(result.history['val_accuracy'], label='Validation Accuracy')
    plt.legend()
    plt.show()

    plt.plot(result.history['loss'], label='Train Loss')
    plt.plot(result.history['val_loss'], label='Validation Loss')
    plt.legend()
    plt.show()

except Exception as e:
    print(f"Error during training: {e}")